## Basic IP Modeling

You're trying to pack as many souvenirs as possible to bring home from your
trip, but your suitcase has a limited capacity. It can hold a maximum of 30 pounds of weight and
15 gallons of volume. Which souvenirs should you pack? The weights and volumes are as follows:


| Souvenir Number | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|-----------------|---|---|---|---|---|---|---|---|---|----|
| Weight          | 5 | 6 | 7 | 6 | 4 | 6 | 7 | 3 | 8 | 5  |
| Volume          | 2 | 4 | 5 | 3 | 3 | 2 | 3 | 1 | 2 | 4  |

### Solution

In [12]:
using JuMP, Gurobi

items = [:1 :2 :3 :4 :5 :6 :7 :8 :9 :10]
weight = Dict(zip(items,[5 6 7 6 4 6 7 3 8 5]))
vol = Dict(zip(items, [2 4 5 3 3 2 3 1 2 4]))

m = Model(solver=GurobiSolver(OutputFlag=0))
    
# binary variables correspond to whether we choose each item
@variable(m, z[items], Bin) 

@objective(m, Max, sum(z))

@constraint(m, sum(z[i]*weight[i] for i in items) <= 30)
@constraint(m, sum(z[i]*vol[i] for i in items) <= 15)

solve(m)
println("We can bring ", getobjectivevalue(m), " items")
println("choose the following items: ")
for i in items
    if getvalue(z[i]) >= 10e-5
        println(i)
    end
end

Academic license - for non-commercial use only
We can bring 6.0 items
choose the following items: 
1
4
5
6
8
10


## Fixed cost practice

Comquat owns four production plants at which personal computers are
produced. Comquat can sell up to 20,000 computers per year at a price of \$3,500 per computer. For
each plant the production capacity, cost per computer, and fixed cost of operating the plant for a year
are given below. Determine how Comquat can maximize its yearly profit from computer production.

| Plant | Production capacity | Fixed Cost (\$ Million) | Per computer cost (\$) |
|-------|---------------------|-------------------------|------------------------|
| 1     | 10,000              | 9                       | 1,000                  |
| 2     | 8,000               | 5                       | 1,700                  |
| 3     | 9,000               | 3                       | 2,300                  |
| 4     | 6,000               | 1                       | 2,900                  |

### Solution

In [25]:
plants = [:1 :2 :3 :4]
capacity = Dict(zip(plants, 1000*[10 8 9 6]))
fixed_cost = Dict(zip(plants, [9 5 3 1]))
var_cost = Dict(zip(plants, 1000*[1 1.7 2.3 2.9]))


using JuMP, Gurobi

m = Model(solver=GurobiSolver(OutputFlag=0))

# production at each plant
@variable(m, x[plants] >= 0)
# whether we open each plant
@variable(m, z[plants], Bin)

# max profit = revenue - fixed cost - variable cost
@objective(m, Max, 3500*sum(x) - 
    1e6*sum(fixed_cost[i] * z[i] for i in plants) - 
    sum(var_cost[i]*x[i] for i in plants))

# max production
@constraint(m, sum(x) <= 20000)

# max prod at each plant
@constraint(m, prod[i in plants], x[i] <= capacity[i])

# can't produce from unused plant: x>0 => z=1
# (note this and the previous constraint can be combined)
@constraint(m, bigM[i in plants], x[i] <= capacity[i]*z[i])

solve(m)
println("Profit: \$", getobjectivevalue(m))
println("We should open plants: ")
for i in plants
    if getvalue(z[i]) > 10e-4
        println(i)
    end
end

Academic license - for non-commercial use only
Profit: $2.56e7
We should open plants: 
1
2
4
